In [22]:
from langchain.agents import create_sql_agent, create_react_agent, Tool, AgentExecutor, AgentType
from langchain.chat_models import ChatOpenAI
from langchain.sql_database import SQLDatabase
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain.utilities import GoogleSerperAPIWrapper
from langchain.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.document_loaders import PyPDFLoader
from langchain.tools.retriever import create_retriever_tool
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_experimental.utilities import PythonREPL

from langchain_core.tools import tool

from typing import Annotated

from langchain import hub

from dotenv import load_dotenv
from langchain.chat_models import ChatOllama


load_dotenv()


# SQL Agent 생성
llm = ChatOpenAI(
    temperature=0.1, 
    model_name='gpt-4o-mini', 
)

# llm = ChatOllama(
#     model="llama3.1:latest",
#     temperature=0.1, 
# )

template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question. and TRANSLATE to korean

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)


prompt2=ChatPromptTemplate.from_messages(
    [
        ("system", "너는 테스트 챗봇이고 한국어로 답해줘.."),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

prompt3 = hub.pull("hwchase17/openai-tools-agent")

search = GoogleSerperAPIWrapper()


# @tool
# def search_tool() -> Tool:
#     """웹 검색이 필요할 경우 이 도구를 사용합니다."""
    
#     search = GoogleSerperAPIWrapper()    

#     return Tool(
#         name="Google_Search",
#         func=search.run,
#         description="웹 검색이 필요할 때 사용합니다.",
#         verbose=True
#     )

@tool
def get_db_tool(db_path) -> Tool:
    """영화(Movie)와 관련된 내용을 찾을 때 이 도구를 사용해야 합니다. 이 도구는 데이터베이스를 조회합니다."""
    
    # 데이터베이스 설정
    db = SQLDatabase.from_uri(db_path)
    toolkit = SQLDatabaseToolkit(db=db, llm=llm)
    
    # SQL Agent 생성 
    sql_agent = create_sql_agent(
        llm=llm,
        toolkit=toolkit,
        agent_type=AgentType.OPENAI_FUNCTIONS,
        verbose=True,
    )

    return Tool(
        name="Database_Search_for_movie",
        func=sql_agent.run,
        description="영화 관련 내용을 찾을 때 사용합니다.",
        verbose=True
    )


@tool
def get_pdf_tool(pdf_path) -> Tool:
    """PDF에서 검색해야할 경우 이 도구를 사용해야 합니다"""
    
    # loader = PyPDFLoader("SPRi AI Brief_8월호_산업동향.pdf")
    loader = PyPDFLoader(pdf_path)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    split_docs = loader.load_and_split(text_splitter)
    
    vector = FAISS.from_documents(split_docs, OpenAIEmbeddings())
    retriever = vector.as_retriever()

    tool = create_retriever_tool(
        retriever,
        name="pdf_search",
        description="'PDF에서 검색해야할 경우 이 도구를 사용해야 합니다",
    )
    
    return tool

repl = PythonREPL()

@tool
def python_repl(
    code: Annotated[str, "The python code to execute to generate your chart."]
):
    """Use this to execute python code. If you want to see the output of a value,
    you should print it out with `print(...)`. This is visible to the user."""
    try:
        result = repl.run(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"
    return f"Succesfully executed:\n```python\n{code}\n```\nStdout: {result}"

agent_tools = [
    Tool(
        name="Google_Search",
        func=search.run,
        description="웹 검색이 필요할 때 사용합니다.",
        verbose=True
    ),
    get_db_tool("sqlite:///movies.sqlite"),
    get_pdf_tool("SPRI_AI_Brief_2023년12월호_F.pdf"),
    python_repl,
]

# React Agent 생성
react_agent = create_react_agent(
    llm, 
    agent_tools, 
    prompt, 
)

agent_executor = AgentExecutor(
    agent=react_agent,
    tools=agent_tools,
    handle_parsing_errors=True,
    verbose=True,
    # return_intermediate_steps=True,
)

########## 6. 채팅 기록을 수행하는 메모리를 추가합니다. ##########

# 채팅 메시지 기록을 관리하는 객체를 생성합니다.
message_history = ChatMessageHistory()

# 채팅 메시지 기록이 추가된 에이전트를 생성합니다.
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    # 대부분의 실제 시나리오에서 세션 ID가 필요하기 때문에 이것이 필요합니다
    # 여기서는 간단한 메모리 내 ChatMessageHistory를 사용하기 때문에 실제로 사용되지 않습니다
    lambda session_id: message_history,
    # 프롬프트의 질문이 입력되는 key: "input"
    input_messages_key="input",
    # 프롬프트의 메시지가 입력되는 key: "chat_history"
    history_messages_key="chat_history",
)

########## 7. 질의-응답 테스트를 수행합니다. ##########

# 질의에 대한 답변을 출력합니다.
response = agent_with_chat_history.invoke(
    {
        "input": "다음 데이터로 차트를 그려줘 '1,2,3,4,5,6,7'"
    },
    # 세션 ID를 설정합니다.
    # 여기서는 간단한 메모리 내 ChatMessageHistory를 사용하기 때문에 실제로 사용되지 않습니다
    config={"configurable": {"session_id": "MyTestSessionID"}},
)
print(f"답변: {response['output']}") 


        # "input": "YouTube 2024년부터 AI 생성콘텐츠 표시 의무화에 대한 내용을 PDF 문서에서 알려줘" 
        # "input": "DB에서 예산이 가장 많이 들어간 영화의 주연배우를 알려줘. 주연배우를 못찾으면 웹에서 찾아줘"
        # "input": "다음 데이터로 차트를 그려줘 '1,2,3,4,5,6,7'"
        

Failed to get info from https://api.smith.langchain.com: LangSmithConnectionError('Connection error caused failure to GET /info  in LangSmith API. Please confirm your internet connection.. ConnectionError(MaxRetryError("HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /info (Caused by NewConnectionError(\'<urllib3.connection.HTTPSConnection object at 0x30a5dbd90>: Failed to establish a new connection: [Errno 8] nodename nor servname provided, or not known\'))"))')


InvalidVersion: Invalid version: ''

In [1]:
import matplotlib.font_manager as fm

# 설치된 폰트 출력
font_list = [font.name for font in fm.fontManager.ttflist]
font_list

['STIXNonUnicode',
 'DejaVu Sans Mono',
 'cmex10',
 'DejaVu Sans',
 'STIXNonUnicode',
 'STIXSizeThreeSym',
 'DejaVu Sans Mono',
 'DejaVu Sans',
 'STIXGeneral',
 'STIXSizeFiveSym',
 'STIXSizeThreeSym',
 'DejaVu Serif',
 'cmtt10',
 'STIXSizeOneSym',
 'STIXSizeFourSym',
 'cmss10',
 'cmr10',
 'STIXGeneral',
 'STIXSizeTwoSym',
 'STIXSizeOneSym',
 'STIXSizeTwoSym',
 'STIXGeneral',
 'DejaVu Sans Mono',
 'DejaVu Sans',
 'DejaVu Sans Display',
 'cmmi10',
 'DejaVu Serif',
 'DejaVu Sans',
 'DejaVu Sans Mono',
 'STIXNonUnicode',
 'STIXSizeFourSym',
 'DejaVu Serif Display',
 'DejaVu Serif',
 'STIXGeneral',
 'STIXNonUnicode',
 'cmsy10',
 'cmb10',
 'DejaVu Serif',
 'Yuanti SC',
 '.SF Arabic',
 'Noto Sans Samaritan',
 'Noto Sans Kaithi',
 'Arial Unicode MS',
 'Tamil MN',
 'Hei',
 'Noto Sans Newa',
 'PT Mono',
 'Big Caslon',
 'Noto Sans Warang Citi',
 'Noto Sans Duployan',
 'Apple Braille',
 'Farisi',
 'Noto Sans Yi',
 'Superclarendon',
 'Raanana',
 '.SF NS Mono',
 'Mukta Mahee',
 'Noto Sans Khudawadi'